In [ ]:
from dfg_ms_plexus.training_setup import *

In [ ]:
# root = Path('F:/DATA/dfg_plexus/')
root = Path('/media/yannik/Intenso/DATA/dfg_plexus')

# df = pd.read_csv(root / "radiomics_features___combined.csv", delimiter=";")
# df = pd.read_csv(root / 'radiomics_and_cnn_features___combined.csv', delimiter=';')
# df = pd.read_csv(root / "radiomics_features___combined_SA___normalized.csv", delimiter=";")
df = pd.read_csv(root / "radiomics_features___harmonized___combined___SA___normalized_ICV.csv", delimiter=";")

# ft = df.drop(columns=['label', 'patID'])
ft = df.drop(columns=['label', 'patID', 'DWI (0no,1yes)', 'LesionVolume', 'DiseaseDuration', 'EDSS'])
label = df['label'].astype(int)
label, class_mapping = get_labels_hc_cis_ms(label)

print(f"{df.shape=}")
print(f"{ft.shape=}")
print(f"{label.shape=}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(ft, label, test_size=0.3, random_state=42, stratify=label)
X_train = X_train.values
y_train = y_train.values
X_test = X_test.values
y_test = y_test.values
print(f"{X_train.shape=}")
print(f"{type(X_train)=}")

In [ ]:
# Preprocess data with Feature Selection and SMOTE

from imblearn.over_sampling import SMOTE
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import RobustScaler

var_threshold = VarianceThreshold(0.001)
X_train_pruned = var_threshold.fit_transform(X_train)
X_test_pruned = var_threshold.fit_transform(X_test)

# 1. Scale the data (Deep learning models prefer scaled data)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_pruned)
X_test_scaled = scaler.transform(X_test_pruned)

# 2. Select the top features (e.g., top 20) to reduce noise
selector = SelectKBest(score_func=f_classif, k=20)
X_train_selected = selector.fit_transform(X_train_scaled, y_train)
X_test_selected = selector.transform(X_test_scaled)

# 3. Apply SMOTE ONLY to the training data to boost Class 1
#smote = SMOTE(random_state=42)
#X_train_resampled, y_train_resampled = smote.fit_resample(X_train_selected, y_train)

print(f"Original training shape: {X_train.shape}")
# print(f"Resampled training shape: {X_train_resampled.shape}")

#X_train = X_train_resampled
#y_train = y_train_resampled

X_train_full, X_test_full = X_train_scaled, X_test_scaled
# X_train_full, X_test= X_train_selected, X_test_selected
y_train_full = y_train
y_test_full = y_test

In [ ]:
import os
import random

def seed_everything(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
import torch
from pytorch_tabnet.pretraining import TabNetPretrainer
from pytorch_tabnet.tab_model import TabNetClassifier

seeds = [0, 1, 2, 3, 4]

all_train_reports = []
all_test_reports = []
all_train_cms = []
all_test_cms = []
all_best_epochs = []
all_best_scores = []

labels_sorted = np.sort(np.unique(label))
class_names = list(class_mapping.keys())

for seed in seeds:
    print(f"\n========== Seed {seed} ==========")

    seed_everything(seed)

    # Inner train/validation split for TabNet early stopping
    X_train_seeded, X_val_seeded, y_train_seeded, y_val_seeded = train_test_split(
        X_train_full,
        y_train_full,
        test_size=0.2,
        random_state=seed,
        stratify=y_train_full,
    )

    print(f"{X_train_seeded.shape=}, {X_val_seeded.shape=}, {X_test_full.shape=}")

    # Unsupervised pretraining
    unsupervised_model = TabNetPretrainer(
        optimizer_fn=torch.optim.Adam,
        optimizer_params=dict(lr=1e-3),
        mask_type='entmax', # "sparsemax"
        seed=seed
    )

    unsupervised_model.fit(
        X_train=X_train_seeded,
        eval_set=[X_val_seeded],
        pretraining_ratio=0.5,  # 0.8
        batch_size=16,
        patience=30,
        max_epochs=1000
    )

    clf = TabNetClassifier(
        n_d=8, n_a=8,         # Shrinking network width (default is 8),
        n_steps=3,              # Keeping the decision steps shallow
        gamma=1.3,              # Higher gamma encourages stronger feature sparsity
        lambda_sparse=1e-5,     # Penalize the use of too many features
        optimizer_fn=torch.optim.AdamW,
        optimizer_params=dict(lr=1e-2, weight_decay=1e-5),
        scheduler_params={"step_size":20, "gamma":0.9},
        scheduler_fn=torch.optim.lr_scheduler.StepLR,
        mask_type='entmax', # This will be overwritten if using pretrain model
        seed=seed
    )

    clf.fit(
        X_train=X_train_seeded, y_train=y_train_seeded,
        eval_set=[(X_train_seeded, y_train_seeded), (X_val_seeded, y_val_seeded)],
        eval_name=['train', 'valid'],
        eval_metric=['balanced_accuracy', 'logloss'],  # (last metric name is used for early stopping)
        # from_unsupervised=unsupervised_model,
        weights=1,              # Automatically balancing classes via sampling
        batch_size=16,
        virtual_batch_size=8,   # Ghost Batch Norm (must be <= batch_size)
        patience=50,
        max_epochs=1000
    )

    all_best_epochs.append(clf.best_epoch)
    all_best_scores.append(clf.best_cost)

    print(f"Best epoch: {clf.best_epoch}")
    print(f"Best validation score: {clf.best_cost:.4f}")

    y_train_pred = clf.predict(X_train_full)
    y_test_pred = clf.predict(X_test_full)

    train_report = classification_report(
        y_train_full,
        y_train_pred,
        labels=labels_sorted,
        target_names=class_names,
        output_dict=True,
        zero_division=0,
    )

    test_report = classification_report(
        y_test_full,
        y_test_pred,
        labels=labels_sorted,
        target_names=class_names,
        output_dict=True,
        zero_division=0,
    )

    all_train_reports.append(report_to_df(train_report))
    all_test_reports.append(report_to_df(test_report))

    all_train_cms.append(
        confusion_matrix(y_train, y_train_pred, labels=labels_sorted)
    )

    all_test_cms.append(
        confusion_matrix(y_test, y_test_pred, labels=labels_sorted)
    )

In [ ]:
train_summary = aggregate_reports(all_train_reports)
test_summary = aggregate_reports(all_test_reports)

print("\n--- Train Set: Mean ± Std over seeds ---")
display(train_summary)

print("\n--- Test Set: Mean ± Std over seeds ---")
display(test_summary)

In [ ]:
train_cms = np.asarray(all_train_cms)
test_cms = np.asarray(all_test_cms)

train_cms_norm = np.asarray([normalize_cm(cm) for cm in train_cms])
test_cms_norm = np.asarray([normalize_cm(cm) for cm in test_cms])

train_cm_mean = train_cms_norm.mean(axis=0)
train_cm_std = train_cms_norm.std(axis=0)

test_cm_mean = test_cms_norm.mean(axis=0)
test_cm_std = test_cms_norm.std(axis=0)

plot_mean_std_cm(
    train_cm_mean,
    train_cm_std,
    "Trainset Confusion Matrix: Mean ± Std over seeds",
    class_names,
)

plot_mean_std_cm(
    test_cm_mean,
    test_cm_std,
    "Testset Confusion Matrix: Mean ± Std over seeds",
    class_names,
)